# UC04 — Mantenimiento Predictivo de Equipos de Planta**Objetivo:** Detectar anomalías en equipos críticos (bombas, soplantes) para predecir fallos con 7-14 días de antelación.**Tecnologías:** ML.ANOMALY_DETECTION · Streamlit**Tiempo estimado:** 15 minutos

## Paso 1 — Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS MANTENIMIENTO_PREDICTIVO_DB;USE DATABASE MANTENIMIENTO_PREDICTIVO_DB;USE SCHEMA PUBLIC;CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;USE WAREHOUSE COMPUTE_WH;

## Paso 2 — Generar Inventario de Equipos50 equipos rotativos: bombas, soplantes, agitadores y compresores.

In [ ]:
CREATE OR REPLACE TABLE equipos ASSELECT    CASE        WHEN SEQ4() < 20 THEN 'BOM-' || LPAD((SEQ4()+1)::STRING, 3, '0')        WHEN SEQ4() < 35 THEN 'SOP-' || LPAD((SEQ4()-19)::STRING, 3, '0')        WHEN SEQ4() < 45 THEN 'AGI-' || LPAD((SEQ4()-34)::STRING, 3, '0')        ELSE 'COM-' || LPAD((SEQ4()-44)::STRING, 3, '0')    END AS equipo_id,    CASE WHEN SEQ4() < 20 THEN 'Bomba' WHEN SEQ4() < 35 THEN 'Soplante' WHEN SEQ4() < 45 THEN 'Agitador' ELSE 'Compresor' END AS tipo,    CASE MOD(SEQ4(), 4) WHEN 0 THEN 'Pretratamiento' WHEN 1 THEN 'Tratamiento' WHEN 2 THEN 'Fangos' ELSE 'Bombeo' END AS zona_planta,    (15 + UNIFORM(0, 135, RANDOM()))::INT AS potencia_kw,    DATEADD('year', -UNIFORM(1, 20, RANDOM()), CURRENT_DATE()) AS fecha_instalacion,    UNIFORM(5000, 80000, RANDOM()) AS horas_acumuladasFROM TABLE(GENERATOR(ROWCOUNT => 50));SELECT tipo, COUNT(*) AS cantidad, ROUND(AVG(potencia_kw),0) AS potencia_media FROM equipos GROUP BY tipo;

## Paso 3 — Generar Datos de Sensores IoT365 días de lecturas diarias por equipo. 5 equipos tienen degradación progresiva en los últimos 30 días.

In [ ]:
CREATE OR REPLACE TABLE sensores_equipos ASSELECT    e.equipo_id,    DATEADD('day', -f.VALUE::INT, CURRENT_DATE()) AS fecha,    ROUND(CASE WHEN e.equipo_id IN ('BOM-001','BOM-005','SOP-003','AGI-002','COM-001') AND f.VALUE::INT < 30        THEN 3 + (30 - f.VALUE::INT) * 0.3 + UNIFORM(0, 20, RANDOM()) * 0.1        ELSE 2 + UNIFORM(0, 30, RANDOM()) * 0.1 END, 2) AS vibracion_mm_s,    ROUND(CASE WHEN e.equipo_id IN ('BOM-001','BOM-005','SOP-003','AGI-002','COM-001') AND f.VALUE::INT < 30        THEN 50 + (30 - f.VALUE::INT) * 1.0 + UNIFORM(0, 5, RANDOM())        ELSE 45 + UNIFORM(0, 20, RANDOM()) END, 1) AS temperatura_rodamiento_c,    ROUND(3.5 + UNIFORM(-5, 10, RANDOM()) * 0.1, 2) AS presion_descarga_bar,    ROUND(CASE WHEN e.equipo_id IN ('BOM-001','BOM-005','SOP-003','AGI-002','COM-001') AND f.VALUE::INT < 30        THEN 85 + (30 - f.VALUE::INT) * 0.5 + UNIFORM(0, 3, RANDOM())        ELSE 70 + UNIFORM(0, 15, RANDOM()) END, 1) AS amperaje_pct_nominal,    UNIFORM(4, 24, RANDOM()) AS horas_operacion_diaFROM equipos e,     LATERAL FLATTEN(ARRAY_GENERATE_RANGE(0, 365)) f;SELECT COUNT(*) AS registros_totales, COUNT(DISTINCT equipo_id) AS equipos FROM sensores_equipos;

## Paso 4 — Entrenar Modelo de Detección de Anomalías

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.ANOMALY_DETECTION detector_anomalias_equipos(    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'sensores_equipos'),    SERIES_COLNAME => 'equipo_id',    TIMESTAMP_COLNAME => 'fecha',    TARGET_COLNAME => 'vibracion_mm_s',    LABEL_COLNAME => '');

## Paso 5 — Detectar Anomalías RecientesAnalizamos los últimos 30 días para identificar equipos con comportamiento anómalo.

In [ ]:
CALL detector_anomalias_equipos!DETECT_ANOMALIES(    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'sensores_equipos'),    SERIES_COLNAME => 'equipo_id',    TIMESTAMP_COLNAME => 'fecha',    TARGET_COLNAME => 'vibracion_mm_s',    CONFIG_OBJECT => {'prediction_interval': 0.99});CREATE OR REPLACE TABLE anomalias_detectadas ASSELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))WHERE IS_ANOMALY = TRUE;SELECT SERIES AS equipo_id, COUNT(*) AS dias_anomalos FROM anomalias_detectadas GROUP BY 1 ORDER BY 2 DESC;

## Paso 6 — Dashboard de Salud de Equipos

In [ ]:
import streamlit as stfrom snowflake.snowpark.context import get_active_sessionsession = get_active_session()st.title("🔧 Mantenimiento Predictivo — Salud de Equipos")col1, col2, col3 = st.columns(3)total_eq = session.sql("SELECT COUNT(DISTINCT equipo_id) AS n FROM equipos").collect()[0]["N"]anomalos = session.sql("SELECT COUNT(DISTINCT SERIES) AS n FROM anomalias_detectadas").collect()[0]["N"]col1.metric("Equipos Monitorizados", total_eq)col2.metric("Equipos con Anomalía", anomalos, delta=f"-{anomalos}", delta_color="inverse")col3.metric("Días Monitorización", "365")st.subheader("Equipos con Más Días Anómalos")df = session.sql("SELECT SERIES AS equipo, COUNT(*) AS dias_anomalos FROM anomalias_detectadas GROUP BY 1 ORDER BY 2 DESC LIMIT 10").to_pandas()st.bar_chart(df.set_index("EQUIPO"))

## Paso 7 — Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS MANTENIMIENTO_PREDICTIVO_DB;